In [13]:
import pandas as pd 


df = pd.read_csv('Data/merged_nord_with_power.csv')

In [14]:
df.head()

,Date and Time,TMP indoor espacesalaries_s41,HUM indoor espacesalaries_s41,CO2 indoor espacesalaries_s41,VOCT indoor espacesalaries_s41,DBAA indoor espacesalaries_s41,DBAP indoor espacesalaries_s41,LIGHT_LUX indoor espacesalaries_s41,OCCUPANCY_RATE indoor espacesalaries_s41,TMP indoor b20,...,CO2 indoor t12,VOCT indoor t12,DBAA indoor t12,DBAP indoor t12,LIGHT_LUX indoor t12,OCCUPANCY_RATE indoor t12,Horodatage_Début,Valeur,Consommation,Horodatage_Fin
0,2026-03-01 00:00:00,17.2,55.0,461.0,16.0,36.0,39.0,0.0,0.0,17.2,...,521.0,16.0,37.0,45.0,0.0,0.0,2026-03-01 00:00:00,39.0,3.250000,2026-03-01 00:10:00
1,2026-03-01 00:10:00,17.2,54.0,452.0,20.0,36.0,39.0,0.0,0.0,17.2,...,529.0,20.0,36.0,40.0,0.0,0.0,2026-03-01 00:10:00,38.0,3.166667,2026-03-01 00:20:00
2,2026-03-01 00:20:00,17.2,54.0,442.0,32.0,37.0,40.0,0.0,0.0,17.2,...,518.0,21.0,38.0,49.0,0.0,0.0,2026-03-01 00:20:00,27.0,2.250000,2026-03-01 00:30:00
3,2026-03-01 00:30:00,17.2,54.0,446.0,28.0,37.0,46.0,0.0,0.0,17.2,...,519.0,19.0,37.0,44.0,0.0,0.0,2026-03-01 00:30:00,16.0,1.333333,2026-03-01 00:40:00
4,2026-03-01 00:40:00,17.1,54.0,453.0,20.0,36.0,42.0,0.0,0.0,NaN,...,507.0,29.0,37.0,45.0,0.0,0.0,2026-03-01 00:40:00,23.0,1.916667,2026-03-01 00:50:00


In [15]:
df.isnull().sum()

Date and Time                       0
TMP indoor espacesalaries_s41     123
HUM indoor espacesalaries_s41     123
CO2 indoor espacesalaries_s41     123
VOCT indoor espacesalaries_s41    123
                                 ... 
OCCUPANCY_RATE indoor t12         251
Horodatage_Début                    0
Valeur                              0
Consommation                        0
Horodatage_Fin                      0
Length: 225, dtype: int64

## 1. Setup PyTorch Environment
Import essential PyTorch libraries and ensure CUDA access if available.

In [16]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

AttributeError: module 'numpy' has no attribute '_no_nep50_warning'

## 2. Prepare Data
Handle missing values, isolate the `Consommation` target variable, and scale the features symmetrically using `StandardScaler`.

In [ ]:
# Drop datetime strings as neural networks need floating point matrices
cols_to_drop = ['Date and Time', 'Horodatage_Début', 'Horodatage_Fin']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df_numeric = df.drop(columns=cols_to_drop)

# Forward fill then backward fill nicely handles sequence missing data gaps!
df_numeric = df_numeric.ffill().bfill()

# Discover Target Column Index
target_col_name = 'Consommation'
if target_col_name not in df_numeric.columns:
    target_col_name = 'Valeur'
target_idx = df_numeric.columns.get_loc(target_col_name)

# Standardize to 0 mean and 1 variance
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_numeric.values)
print(f"Transformed sequence shape: {data_scaled.shape}")

## 3. Dataset and DataLoader
Windowing the time-series arrays and batching them for optimal LSTM sequence prediction! By leveraging `Subsets`, we avoid sequential leakage over our chronological timeline that typical `random_split` brings.

In [ ]:
class PowerConsumptionDataset(Dataset):
    def __init__(self, data, target_col_idx, seq_length=12):
        self.data = data
        self.target_col_idx = target_col_idx
        self.seq_length = seq_length
        
    def __len__(self):
        return len(self.data) - self.seq_length
        
    def __getitem__(self, idx):
        # Input history (e.g. past 2 hours block)
        x = self.data[idx : idx + self.seq_length, :]
        # Output target prediction (next value in line!)
        y = self.data[idx + self.seq_length, self.target_col_idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

seq_length = 12 
dataset = PowerConsumptionDataset(data_scaled, target_col_idx=target_idx, seq_length=seq_length)

# 80/20 chronological validation splitting
train_size = int(0.8 * len(dataset))
train_dataset = torch.utils.data.Subset(dataset, range(0, train_size))
val_dataset = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 4. LSTM Architecture
A powerful recurrent architecture to learn memory from historical sensor/power dynamics, resolving common gradient loss patterns!

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.rnn(x, (h0, c0))
        # Keep solely the outcome generated globally post sequence inference
        out = self.fc(out[:, -1, :])
        return out

input_size = data_scaled.shape[1]
hidden_size = 64
num_layers = 2

model = RNNModel(input_size, hidden_size, num_layers).to(device)
print(model)

## 5. Iteration Engine
Initialize parameters, process sequential matrices efficiently across our computation `device`, capture backward gradients to mathematically hone errors!

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20

train_losses = []
val_losses = []

print(f"Starting neural convergence loop on {device}...")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * X_batch.size(0)
        
    train_loss /= len(train_dataset)
    train_losses.append(train_loss)
    
    # Eval Step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            val_loss += loss.item() * X_batch.size(0)
            
    val_loss /= len(val_dataset)
    val_losses.append(val_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:02d}/{num_epochs}], Train MSE: {train_loss:.4f}, Val MSE: {val_loss:.4f}')

print("Training fully concluded!")

## 6. Metric Tracking Graphs
Let's review how closely the model adjusted to our training sequence!

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training MSE', color='blue')
plt.plot(val_losses, label='Validation MSE', color='orange')
plt.title('LSTM Sequence MSE Validation Journey')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error Loss')
plt.legend()
plt.grid(True)
plt.show()